# Station character — what *kind of place* is each Toronto bike station?

Every station has a personality revealed by its arrival/departure rhythm:
- **Residential:** bikes leave in the morning, return in the evening (residents commute out).
- **Workplace:** bikes arrive in the morning, depart in the evening.
- **Leisure:** traffic peaks on weekends and midday, not 9-to-5.
- **Nightlife:** activity leans late evening.
- **Balanced / hub:** heavy traffic that doesn't favor a direction.

We infer these from 2025 trip timestamps alone, then colour every station on a Toronto map according to its character. Goal: a functional-urbanism map of the city drawn purely by how people move bikes.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import duckdb
import numpy as np
import pandas as pd
import pydeck as pdk

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROC = Path.cwd().parent / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)

GLOB = str(DATA_RAW / 'ridership' / '2025' / '*.csv')
con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{GLOB}', header=True, ignore_errors=true, union_by_name=true)")

stations = pd.read_parquet(DATA_RAW / 'stations.parquet')
stations['station_id_int'] = pd.to_numeric(stations['station_id'], errors='coerce').astype('Int64')
station_lookup = stations.dropna(subset=['station_id_int']).set_index('station_id_int')[['name', 'lat', 'lon']]

## 1. Build arrival / departure matrix per station

For each station, count trips that *depart* (row: as origin) and *arrive* (row: as destination), broken out by `hour of day` (0-23) and `is_weekend`.

In [2]:
# Departures: group by Start_Station_Id.
dep = con.execute('''
    SELECT Start_Station_Id AS station_id,
           EXTRACT(hour FROM Start_Time) AS hour,
           (EXTRACT(dow FROM Start_Time) IN (0, 6)) AS is_weekend,
           COUNT(*) AS departures
    FROM trips
    WHERE Start_Station_Id IS NOT NULL AND Start_Time IS NOT NULL
    GROUP BY 1, 2, 3
''').fetchdf()

# Arrivals: group by End_Station_Id (TRUSTED — unlike End_Station_Name).
arr = con.execute('''
    SELECT End_Station_Id AS station_id,
           EXTRACT(hour FROM End_Time) AS hour,
           (EXTRACT(dow FROM End_Time) IN (0, 6)) AS is_weekend,
           COUNT(*) AS arrivals
    FROM trips
    WHERE End_Station_Id IS NOT NULL AND End_Time IS NOT NULL
    GROUP BY 1, 2, 3
''').fetchdf()

print('dep rows:', len(dep), 'arr rows:', len(arr))
dep.head()

dep rows: 45382 arr rows: 45815


,station_id,hour,is_weekend,departures
0,7248,15,False,1350
1,7391,14,False,968
2,7272,15,False,763
3,7462,15,False,2050
4,7724,14,False,366


In [3]:
# Pivot to long-form merged table (station_id, hour, is_weekend, departures, arrivals).
flow = dep.merge(arr, on=['station_id', 'hour', 'is_weekend'], how='outer').fillna(0)
flow['departures'] = flow['departures'].astype(int)
flow['arrivals']   = flow['arrivals'].astype(int)
flow['net']        = flow['arrivals'] - flow['departures']  # +ve = fills up, -ve = empties
flow.head()

,station_id,hour,is_weekend,departures,arrivals,net
0,7000,0,False,186,417,231
1,7000,0,True,211,366,155
2,7000,1,False,78,191,113
3,7000,1,True,115,298,183
4,7000,2,False,67,88,21


## 2. Derive character metrics per station

Simple, interpretable metrics:

- **`commuter_score`** = weekday AM outflow (7-10am departures − arrivals) minus weekday PM outflow (4-7pm departures − arrivals). Positive → residential (bikes leave in morning, return in evening). Negative → workplace (opposite).
- **`weekend_lift`** = weekend trips / weekday trips, normalized to null-hypothesis ratio of 2/5. `>1` → weekend-dominant.
- **`evening_lean`** = fraction of weekday activity happening after 8pm.
- **`volume`** = total trips at this station all year.

In [4]:
wk = flow[~flow['is_weekend']].copy()
we = flow[flow['is_weekend']].copy()

def window_net(df: pd.DataFrame, hours: range) -> pd.Series:
    sub = df[df['hour'].isin(hours)]
    agg = sub.groupby('station_id')[['departures', 'arrivals']].sum()
    # net outflow: departures - arrivals (positive = station empties during window)
    return agg['departures'] - agg['arrivals']

am_out = window_net(wk, range(7, 10))    # 7,8,9
pm_out = window_net(wk, range(16, 19))   # 16,17,18

wk_volume = wk.groupby('station_id')[['departures', 'arrivals']].sum().sum(axis=1)  # total trips (arr+dep) weekday
we_volume = we.groupby('station_id')[['departures', 'arrivals']].sum().sum(axis=1)
total_vol = wk_volume.add(we_volume, fill_value=0)

late = wk[wk['hour'].isin([20, 21, 22, 23])].groupby('station_id')[['departures', 'arrivals']].sum().sum(axis=1)
evening_lean = (late / wk_volume).fillna(0)

# Weekend lift: weekend daily rate / weekday daily rate. Null hypothesis ratio ≈ 2/5.
weekend_lift = ((we_volume / 2) / (wk_volume / 5)).replace([np.inf, -np.inf], np.nan)

char = pd.DataFrame({
    'volume': total_vol,
    'am_out': am_out, 'pm_out': pm_out,
    'commuter_score': am_out.sub(pm_out, fill_value=0),
    'evening_lean': evening_lean,
    'weekend_lift': weekend_lift,
}).fillna(0)

char = char.join(station_lookup[['name', 'lat', 'lon']], how='inner')
print(len(char), 'stations with character data')
char.describe()

995 stations with character data


,volume,am_out,pm_out,commuter_score,evening_lean,weekend_lift,lat,lon
count,995.000000,995.000000,995.000000,995.000000,995.000000,995.000000,995.000000,995.000000
mean,14969.025126,19.407035,30.287437,-10.880402,0.166783,1.112052,43.681113,-79.389245
std,17780.420522,2139.329506,1242.206854,3306.208372,0.065513,0.494381,0.042977,0.072970
min,2.000000,-44591.000000,-3896.000000,-62387.000000,0.000000,0.000000,43.588077,-79.603464
25%,1449.000000,-11.000000,-266.500000,-22.500000,0.133073,0.859003,43.650097,-79.431700
50%,7708.000000,38.000000,-18.000000,76.000000,0.161136,1.012522,43.666667,-79.392459
75%,23319.000000,400.000000,40.000000,712.000000,0.194870,1.243541,43.704253,-79.354224
max,111199.000000,5428.000000,19125.000000,8303.000000,1.000000,6.250000,43.812642,-79.123184


## 3. Assign each station a functional type

Rule-based, transparent. Thresholds tuned for interpretability, not cross-validated — the goal is a map that tells a story at a glance.

Order matters: weekend-leisure and nightlife take precedence over commuter labels, since those patterns are stronger signals of purpose.

In [5]:
# Ignore stations with trivial volume — their metrics are noisy.
MIN_VOLUME = 500  # total trips per year (arr+dep)
active = char[char['volume'] >= MIN_VOLUME].copy()
print(f'{len(active)} active stations (>= {MIN_VOLUME} trips/yr)')

# Normalize commuter_score by volume so big hubs don't dominate the classification.
active['commuter_score_norm'] = active['commuter_score'] / active['volume']

def classify(r: pd.Series) -> str:
    if r['weekend_lift'] > 1.4:
        return 'weekend-leisure'
    if r['evening_lean'] > 0.22:
        return 'nightlife'
    if r['commuter_score_norm'] > 0.08:
        return 'residential'
    if r['commuter_score_norm'] < -0.08:
        return 'workplace'
    return 'balanced'

active['character'] = active.apply(classify, axis=1)
print(active['character'].value_counts())

871 active stations (>= 500 trips/yr)
character
balanced           403
residential        196
weekend-leisure    120
workplace           93
nightlife           59
Name: count, dtype: int64


In [6]:
# Peek at each category's exemplars — the most extreme station in each.
for label in ['residential', 'workplace', 'weekend-leisure', 'nightlife', 'balanced']:
    sub = active[active['character'] == label]
    if label == 'residential':
        top = sub.nlargest(5, 'commuter_score_norm')
    elif label == 'workplace':
        top = sub.nsmallest(5, 'commuter_score_norm')
    elif label == 'weekend-leisure':
        top = sub.nlargest(5, 'weekend_lift')
    elif label == 'nightlife':
        top = sub.nlargest(5, 'evening_lean')
    else:
        top = sub.nlargest(5, 'volume')
    print(f'\n=== {label.upper()} exemplars ===')
    cols = ['name', 'volume', 'commuter_score_norm', 'evening_lean', 'weekend_lift']
    print(top[cols].to_string(index=False))


=== RESIDENTIAL exemplars ===
                              name  volume  commuter_score_norm  evening_lean  weekend_lift
Western Battery Rd / Pirandello St 18970.0             0.310121      0.133462      0.776112
  St. Columba Pl / St. Clair Ave E  1765.0             0.269122      0.135802      0.704430
                      Orchard Park  4456.0             0.266382      0.092254      0.731796
                Florence Gell Park  2071.0             0.251569      0.139113      0.979503
           King St W / Crawford St 17850.0             0.242353      0.126175      0.855515

=== WORKPLACE exemplars ===
                             name  volume  commuter_score_norm  evening_lean  weekend_lift
   King St W / Bay St (West Side) 94554.0            -0.659803      0.027311      0.106344
            Temperance St Station 82734.0            -0.583086      0.041334      0.168322
    York St / Wellington St W (2) 10897.0            -0.500688      0.056781      0.282118
           Adelaide St W

## 4. Render the character map

Each station a circle, size = sqrt(volume), color = character. Use a colour palette that *reads* urbanistically:
- red for residential (lived-in, warm)
- blue for workplace (cold, institutional)
- green for leisure
- purple for nightlife
- grey for balanced/hub

In [7]:
PALETTE = {
    'residential':     [215,  70,  70],   # warm red
    'workplace':       [ 40, 110, 210],   # cool blue
    'weekend-leisure': [ 60, 165,  85],   # green
    'nightlife':       [155,  70, 190],   # purple
    'balanced':        [150, 150, 150],   # grey
}

m = active.reset_index().rename(columns={'index': 'station_id'}).copy()
m['color'] = m['character'].map(lambda c: PALETTE[c] + [230])

# 3D column heights: proportional to annual volume, visible at zoom 12.
# Clip extreme hubs so they don't blot out neighbours; map to meters above ground.
v_clip = m['volume'].clip(upper=m['volume'].quantile(0.98))
m['elevation'] = (v_clip / v_clip.max() * 450).round().astype(int)
m['radius'] = 45  # constant column footprint; height carries the magnitude signal

m['volume_k'] = (m['volume'] / 1000).round(1)
m['commuter_pct'] = (m['commuter_score_norm'] * 100).round(1)
m['weekend_lift_r'] = m['weekend_lift'].round(2)
m['evening_pct'] = (m['evening_lean'] * 100).round(1)

columns = pdk.Layer(
    'ColumnLayer',
    data=m[['name', 'lat', 'lon', 'color', 'radius', 'elevation', 'character',
            'volume_k', 'commuter_pct', 'weekend_lift_r', 'evening_pct']],
    get_position=['lon', 'lat'],
    get_elevation='elevation',
    elevation_scale=1,
    get_fill_color='color',
    radius='radius',
    pickable=True,
    auto_highlight=True,
    extruded=True,
    material={'ambient': 0.6, 'diffuse': 0.8, 'shininess': 32, 'specularColor': [255, 255, 255]},
)

view = pdk.ViewState(latitude=43.657, longitude=-79.392, zoom=12.4, pitch=55, bearing=-25)
deck = pdk.Deck(
    layers=[columns],
    initial_view_state=view,
    map_style='light',
    tooltip={'text': '{name}\n{character}  ·  {volume_k}k trips/yr\ncommuter: {commuter_pct}%   weekend lift: {weekend_lift_r}×   late-evening: {evening_pct}%'},
)
out = DATA_PROC / 'character_map.html'
deck.to_html(str(out), notebook_display=False)
print('Wrote 3D character map:', out)

Wrote 3D character map: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/character_map.html


## 5. Tell the story — what do these categories reveal about Toronto?

In [8]:
summary = active.groupby('character').agg(
    n_stations=('name', 'size'),
    total_trips=('volume', 'sum'),
    lat_centroid=('lat', 'mean'),
    lon_centroid=('lon', 'mean'),
).sort_values('n_stations', ascending=False)
summary

,n_stations,total_trips,lat_centroid,lon_centroid
character,,,,
balanced,403,8654668.0,43.668373,-79.398285
residential,196,2754688.0,43.663693,-79.390583
weekend-leisure,120,967144.0,43.689816,-79.392447
workplace,93,2146987.0,43.661609,-79.390306
nightlife,59,344663.0,43.716349,-79.386860


In [9]:
def diverging_color(score: float, cap: float = 0.25) -> list:
    """Map commuter_score_norm in [-cap, +cap] to an RGB triple.
    Positive (residential) -> red. Negative (workplace) -> blue. Zero -> warm off-white."""
    x = max(-1.0, min(1.0, float(score) / cap))
    if x >= 0:
        # off-white (245,235,225) -> red (190,45,55)
        r = int(245 + (190 - 245) * x)
        g = int(235 + ( 45 - 235) * x)
        b = int(225 + ( 55 - 225) * x)
    else:
        x = -x
        # off-white -> blue (35,90,170)
        r = int(245 + ( 35 - 245) * x)
        g = int(235 + ( 90 - 235) * x)
        b = int(225 + (170 - 225) * x)
    return [r, g, b, 235]

g = active.reset_index().rename(columns={'index': 'station_id'}).copy()
g['color'] = g['commuter_score_norm'].map(diverging_color)
g['radius'] = np.clip(np.sqrt(g['volume']) * 1.5, 40, 260)
g['commuter_pct'] = (g['commuter_score_norm'] * 100).round(1)
g['volume_k'] = (g['volume'] / 1000).round(1)

grad_layer = pdk.Layer(
    'ScatterplotLayer',
    data=g[['name', 'lat', 'lon', 'color', 'radius', 'commuter_pct', 'volume_k']],
    get_position=['lon', 'lat'],
    get_radius='radius',
    get_fill_color='color',
    pickable=True,
    auto_highlight=True,
    opacity=0.88,
    stroked=True,
    line_width_min_pixels=0.4,
    get_line_color=[40, 40, 40, 150],
)

grad_view = pdk.ViewState(latitude=43.660, longitude=-79.395, zoom=12.6, pitch=0, bearing=0)
grad_deck = pdk.Deck(
    layers=[grad_layer],
    initial_view_state=grad_view,
    map_style='light',
    tooltip={'text': '{name}\ncommuter score: {commuter_pct}%   ({volume_k}k trips/yr)'},
)
grad_out = DATA_PROC / 'commuter_gradient_map.html'
grad_deck.to_html(str(grad_out), notebook_display=False)
print('Wrote commuter gradient map:', grad_out)

Wrote commuter gradient map: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/commuter_gradient_map.html


## 6. Continuous commuter gradient

Same data, different lens. Every station coloured on a diverging red↔blue scale by its `commuter_score_norm`: deep red = pure residential (bikes leave in morning), deep blue = pure workplace (bikes arrive in morning), off-white = balanced. This trades the interpretability of discrete categories for a continuous geography of Toronto's home-vs-work pattern.